In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

from src.config import *
from src.data import *
from src.preprocessing import learn_preprocessing, apply_preprocessing
from src.models import make_lightgbm

In [3]:
X, y = load_train()
X_tr, X_val, y_tr, y_val = get_split(X, y)
print(X_tr.shape, X_val.shape)

(559708, 21) (139927, 21)


In [4]:
test_ids, X_test = load_test()
print(X_test.shape)

(299844, 21)


In [5]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=SEED
)

test_preds = np.zeros(len(X_test))
oof_preds = np.zeros(len(X))

In [6]:
fold_scores = []

for fold, (train_idx, val_idx) in enumerate(cv.split(X, y), start=1):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    # Learn preprocessing only from this fold's training data
    state = learn_preprocessing(
        X_train,
        ratings="numeric",
        scaling="no"
    )

    X_train_processed = apply_preprocessing(X_train, state)
    X_val_processed = apply_preprocessing(X_val, state)
    X_test_processed = apply_preprocessing(X_test, state)

    # Train the model
    model = make_lightgbm(
        num_leaves=63,
        learning_rate=0.025,
        n_estimators=1000
    )
    model.fit(X_train_processed, y_train)

    # Validation predictions and score
    val_preds = model.predict_proba(X_val_processed)[:, 1]
    oof_preds[val_idx] = val_preds

    score = roc_auc_score(y_val, val_preds)
    fold_scores.append(score)

    # Accumulate test predictions
    test_preds += model.predict_proba(X_test_processed)[:, 1] / cv.n_splits

    print(f"Fold {fold}: AUC = {score:.5f}")

print(f"\nMean AUC: {np.mean(fold_scores):.5f}")
print(f"Std AUC:  {np.std(fold_scores):.5f}")
print(f"OOF AUC:  {roc_auc_score(y, oof_preds):.5f}")

Fold 1: AUC = 0.95881
Fold 2: AUC = 0.95734
Fold 3: AUC = 0.95925
Fold 4: AUC = 0.95853
Fold 5: AUC = 0.95881

Mean AUC: 0.95855
Std AUC:  0.00065
OOF AUC:  0.95854


In [7]:
MODEL_DIR.mkdir(parents=True, exist_ok=True)

joblib.dump({"state": state, "model": model}, MODEL_DIR / "lightgbm_oof.joblib")

['C:\\Users\\Bharat\\Desktop\\remaining-2026\\Career-Tracks\\Track 01 - Data Science\\Relevant Projects\\airline-satisfaction-prediction\\models\\lightgbm_oof.joblib']